# **PHASE 03 — FEATURE ENGINEERING ENGINE**
- For 10 day profit
- let create new table **stocks_features_table** > reads data from table **stocks_sorted_table** > copies all original columns > Adds new calculated columns
- WINDOW w → defines a reusable window that we can use any where insted of typing long query several times,it say defines how rows are grouped and ordered
  - grouped by stock symbol
  - ordered by date
- **We will create columns / features for model training here**
  - wef
 
#### stocks_features_table > stocks_features_clean_table

### Helpers

In [29]:
import os
from datetime import datetime
import numpy as np
import duckdb
import pandas as pd

# ================ CONFIG ===================

pd.set_option('display.max_columns', None)

os.makedirs("log", exist_ok=True) # Create log folder if doesn't exit
# dynamically names a log file using the current date & time
LOG_FILE = os.path.join("log", f"loader_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")

# method to add log msg into log file & prints them to console
def log(msg):
    with open(LOG_FILE, "a", encoding="utf-8") as f: # "a" mode → appends messages instead of overwriting
        f.write(msg + "\n")
    print(msg)

### Create DuckDB Connection

In [30]:
import os
import duckdb

DB_FOLDER = "database"
DB_PATH = os.path.join(DB_FOLDER, "cse_data.db")

con = duckdb.connect(database=DB_PATH)

### Create Features Base Table 

In [31]:
# ============== Create Feature Table Base ===========================================================

log("Creating features base table...")

# ============== Reset tables, cols ==============
con.execute("""
    DROP TABLE IF EXISTS stocks_features_table;
""")


# Creates a table called stocks_features_table, If it already exists → replaces it
# SELECT *,  > Selects all columns from stocks_sorted_table last validt table
# LAG window function looks backward in the result set, returns the value of the close column from 5 rows earlier

# close_5d     --> price 5 day ealier 
# close_10d    --> price 10 ddays earlier, 
# high_20      --> highest closing price from the last 20 rows (including today)
# low_20       --> lowest closing price from the last 20 rows (including today)
# avg_vol_20   --> average volume over the last 20 records including today
# std_vol_20   --> standard deviation, how stable or unstable volume has been recently(how much the trading volume has been changing(varying) over the 
#                  last 20 rows including today) if small : volume is stable if large : volume is fluctuating a lot 
#                  how to cal : Find the average of last 20 rec > Find difference from average of each > Square differences(ignore - vals) 
#                  > Average of squared values > Square root 
con.execute("""
    CREATE OR REPLACE TABLE stocks_features_table AS
    SELECT
        *,
        -- LAG always looks back in time, LAG(close, 1) -> closing price 1 day before today, Create new columns also close_1d
        LAG(close, 1)  OVER w AS close_1d,
        LAG(close, 3)  OVER w AS close_3d,
        LAG(close, 5)  OVER w AS close_5d,
        LAG(close, 10) OVER w AS close_10d,
        LAG(close, 20) OVER w AS close_20d,
        LAG(close, 60) OVER w AS close_60d,
    
        -- look at today + previous 19 days → total 20 days, pick the highest / lowest closing price in that 20-day window
        MAX(close) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS high_20,
        MIN(close) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS low_20,
    
        -- Same 20-day window, measures how volatile the volume is over these 20 days, how much the daily volume fluctuates
        -- only compute when full 20 rows exist
        CASE 
            WHEN COUNT(volume) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) = 20
            THEN STDDEV(volume) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)
            ELSE NULL
        END AS std_vol_20
    FROM stocks_sorted_table
    WINDOW w AS (PARTITION BY symbol ORDER BY date);
""")

df1 = con.execute("SELECT * FROM stocks_features_table").fetch_df()
print("1========== stocks_features_table \n", df1.head(25),"\n")

# df = con.execute("""
# SELECT symbol, COUNT(*) 
# FROM stocks_features_table
# GROUP BY symbol;
# """).fetch_df()
# print("DF", df)

# pd.set_option("display.max_rows", None)
# print(df1[df1["symbol"] == "TAJ.N0000"][["date","close","close_60d"]])

# tables > stocks_features_table(main)
# newly added col > close_1d   close_3d   close_5d   close_10d   close_20d   close_60d   high_20   low_20   std_vol_20

Creating features base table...
1========== stocks_features_table 
                   company      symbol  volume  trades  prev_close     open  \
0   ABANS ELECTRICALS PLC  ABAN.N0000    7508     186     1419.00  1419.00   
1   ABANS ELECTRICALS PLC  ABAN.N0000    5970     192     1491.00  1500.00   
2   ABANS ELECTRICALS PLC  ABAN.N0000    2052      90     1561.25  1600.00   
3   ABANS ELECTRICALS PLC  ABAN.N0000    5545     154     1561.00  1590.00   
4   ABANS ELECTRICALS PLC  ABAN.N0000    1976     125     1533.50  1533.50   
5   ABANS ELECTRICALS PLC  ABAN.N0000    1631      79     1594.25  1595.00   
6   ABANS ELECTRICALS PLC  ABAN.N0000    2734     112     1589.75  1580.00   
7   ABANS ELECTRICALS PLC  ABAN.N0000    1102      42     1564.75  1564.00   
8   ABANS ELECTRICALS PLC  ABAN.N0000     962      58     1560.75  1580.00   
9   ABANS ELECTRICALS PLC  ABAN.N0000    4130     158     1551.50  1579.75   
10  ABANS ELECTRICALS PLC  ABAN.N0000    2647      93     1485.00  1490.00

### Creating Momentum Features

In [32]:
# =================== Creating Momentum Features ==========================================================

# Momentum = how fast price is changing

# Calculates x-day return as (Current price ÷ price x days later from today) − 1 >>> Without -1 : 120 / 100 = 1.2, but we need growth  as 0.2
# ret_1  --> Very short-term momentum
# ret_3  --> 3-day return, Small trend detection
# ret_5  --> 5-day return, Weekly momentum
# ret_10 --> 10-day return, Medium-term trend
# ret_20 --> 20-day return, Monthly trend, strong signal
# ret_60 --> 60-day return, Long-term trend

log("Creating momentum features...")

# Reset tables, cols
cols = ["ret_1","ret_3","ret_5","ret_10","ret_20","ret_60"]
for c in cols:
    con.execute(f"ALTER TABLE stocks_features_table DROP COLUMN IF EXISTS {c}")

# Creating columns/features
con.execute("""
    ALTER TABLE stocks_features_table ADD COLUMN ret_1 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN ret_3 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN ret_5 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN ret_10 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN ret_20 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN ret_60 DOUBLE;
""")

# Adding values to columns/features
con.execute("""
    UPDATE stocks_features_table
    SET 
        ret_1 = (close / close_1d) - 1,
        -- Return (percentage change) over the last 3 days
        ret_3 = (close / close_3d) - 1,
        ret_5 = (close / close_5d) - 1,
        ret_10 = (close / close_10d) - 1,
        ret_20 = (close / close_20d) - 1,
        ret_60 = (close / close_60d) - 1;
""")

df1 = con.execute("SELECT * FROM stocks_features_table").fetch_df()
print("2========== stocks_features_table \n", df1.sample(25),"\n")

# tables > _
# newly added col > ret_1   ret_3  ret_5   ret_10   ret_20   ret_60

Creating momentum features...
2========== stocks_features_table 
                                                  company      symbol   volume  \
10213                         LOLC GENERAL INSURANCE PLC  LGIL.N0000   304809   
14848                     MALWATTE VALLEY PLANTATION PLC   MAL.N0000      402   
13846                             MARAWILA  RESORTS  PLC  MARA.N0000   297846   
5678                              CEYLON COLD STORES PLC   CCS.N0000    41832   
9798                         ASIRI HOSPITAL HOLDINGS PLC  ASIR.N0000    20738   
7792                          WASKADUWA BEACH RESORT PLC  CITW.N0000  2143495   
13226           COLOMBO LAND AND DEVELOPMENT COMPANY PLC  CLND.N0000   274513   
10833                         FIRST CAPITAL HOLDINGS PLC  CFVF.N0000    30323   
8009                          LANKA MILK FOODS (CWE) PLC   LMF.N0000  5487390   
9448                     SENKADAGALA FINANCE COMPANY PLC  SFCL.N0000     1187   
3477                                CEYLON 

### Creating Volatility Features

In [33]:
# =================== Creating Volatility Features ==========================================================

# Capture how “unstable” the price is — important for predicting bigger moves
# Volatility features measure how much the price moves or fluctuates


log("Creating volatility features...\n")

# Reset tables, cols
cols = ["std_close_5","std_close_10","std_close_20","range_5","atr_14","TR"]
for c in cols:
    con.execute(f"ALTER TABLE stocks_features_table DROP COLUMN IF EXISTS {c}")
con.execute("DROP TABLE IF EXISTS temp_tr_table")
con.execute("DROP TABLE IF EXISTS temp_vol")



# Add new columns to main table if they don't exist
con.execute("""
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS std_close_5 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS std_close_10 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS std_close_20 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS range_5 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS atr_14 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS TR DOUBLE;
""")

# Craete a new tempory table (temp_tr_table) for calculating True Range (TR)
# TR - Compute True Range first using a CTE (Common Table Expression)
# TR measures the actual daily price movement, considering possible gaps between yesterday’s close and today’s high/low
# TR captures real volatility including overnight gaps
# TR=max of same day (high−low) OR ∣high−previous close∣ OR ∣low−previous close∣ 
con.execute("""
    CREATE OR REPLACE TABLE temp_tr_table AS
    SELECT 
        symbol,
        date,
        high,
        low,
        close,
        GREATEST(
            high - low,
            ABS(high - LAG(close) OVER (PARTITION BY symbol ORDER BY date)),
            ABS(low - LAG(close) OVER (PARTITION BY symbol ORDER BY date))
        ) AS TR
    FROM stocks_features_table;
""")

# Craete a new tempory table (temp_vol_table) for calculating std_close_5, std_close_10, std_close_20, range_5, atr_14
# Compute volatility features using a CTE, DON'T FETCH ALL COLS THEN IT CAUSE TO DEFINE DUPLICATES WITH _1 SUFFIX FOR COLUMNS WE TRY TO MERGE

# std_close_5 - how much the price has fluctuated within the last 5 days. standard deviation measures how much a set of numbers varies from their average
# range_5 - total price movement over the last 5 days (range_5 = max(high(5days))−min(low(5days)))
# atr_14 - Average True Range over 14 days measures the average size of price movements during the last 14 days.
con.execute("""
    CREATE OR REPLACE TABLE temp_vol_table AS
    SELECT 
        symbol,
        date,
        close,
        -- computes rolling standard deviations of the closing prices, 5, 10, 20d
        CASE WHEN COUNT(close) OVER w_5 = 5 THEN STDDEV(close) OVER w_5 ELSE NULL END AS std_close_5,
        CASE WHEN COUNT(close) OVER w_10 = 10 THEN STDDEV(close) OVER w_10 ELSE NULL END AS std_close_10,
        CASE WHEN COUNT(close) OVER w_20 = 20 THEN STDDEV(close) OVER w_20 ELSE NULL END AS std_close_20,
    
        -- range_5 → max high minus min low over a 5-day window
        MAX(high) OVER w_5 - MIN(low) OVER w_5 AS range_5,
    
        -- atr_14 → average True Range over the last 14 days (a standard measure of volatility)         
        AVG(TR) OVER w_14 AS atr_14
    FROM temp_tr_table
    WINDOW
        -- Same 5, 10, 20, 14-day window, measures how somwething is hapenning over these days
        -- looks at today + previous 4 days → total 5 days
        w_5  AS (PARTITION BY symbol ORDER BY date ROWS BETWEEN 4 PRECEDING AND CURRENT ROW),
        w_10 AS (PARTITION BY symbol ORDER BY date ROWS BETWEEN 9 PRECEDING AND CURRENT ROW),
        w_20 AS (PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW),
        w_14 AS (PARTITION BY symbol ORDER BY date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW);
""")

# Merge temp_tr_table back into original table (Overright TR col in main table)
con.execute("""
    UPDATE stocks_features_table AS s
    SET
        TR = tr.TR
    FROM temp_tr_table AS tr
    WHERE s.symbol = tr.symbol AND s.date = tr.date;
""")

# Merge temp_vol_table back into original table (Overright std_close_5, .... cols in main table)
con.execute("""
    UPDATE stocks_features_table AS s
    SET
        std_close_5 = tv.std_close_5,
        std_close_10 = tv.std_close_10,
        std_close_20 = tv.std_close_20,
        range_5 = tv.range_5,
        atr_14 = tv.atr_14,
    FROM temp_vol_table AS tv
    WHERE s.symbol = tv.symbol AND s.date = tv.date;
""")

df1 = con.execute("SELECT * FROM temp_tr_table").fetch_df() # temp_tr_table  temp_vol_table stocks_features_table
print("3========== temp_tr_table \n", df1.head(25),"\n")

df1 = con.execute("SELECT * FROM temp_vol_table").fetch_df() # temp_tr_table  temp_vol_table stocks_features_table
print("3========== temp_vol_table \n", df1.head(25),"\n")

df1 = con.execute("SELECT * FROM stocks_features_table").fetch_df() # temp_tr_table  temp_vol_table stocks_features_table
print("3========== stocks_features_table \n", df1.head(25),"\n")

# tables > temp_tr_table(TR)   temp_vol_table()
# newly added col > std_close_5   std_close_10   std_close_20   range_5   atr_14   TR

Creating volatility features...

3========== temp_tr_table 
         symbol       date     high      low    close       TR
0   APLA.N0000 2025-10-23  1699.00  1668.00  1676.75    31.00
1   APLA.N0000 2025-10-24  1699.00  1646.25  1657.50    52.75
2   APLA.N0000 2025-10-27  1697.00  1625.00  1637.25    72.00
3   APLA.N0000 2025-10-28  1700.00  1625.00  1677.00    75.00
4   APLA.N0000 2025-10-29  1750.00  1650.00  1718.75   100.00
5   APLA.N0000 2025-10-30  1750.00  1715.00  1730.00    35.00
6   APLA.N0000 2025-10-31  1748.75  1700.00  1708.50    48.75
7   APLA.N0000 2025-11-03  1744.50  1700.00  1700.00    44.50
8   APLA.N0000 2025-11-04  1744.25  1647.50  1660.25    96.75
9   APLA.N0000 2025-11-06  1697.75  1630.00  1665.50    67.75
10  APLA.N0000 2025-11-07  1728.00  1645.00  1672.00    83.00
11  APLA.N0000 2025-11-10  1728.75  1630.25  1643.50    98.50
12  APLA.N0000 2025-11-12  1715.00  1622.00  1694.75    93.00
13  APLA.N0000 2025-11-13  1710.00  1650.00  1655.00    60.00
14  APLA.

### Creating Liquidity Features

In [34]:
# =================== Creating Liquidity Features =========================================================

# how easy it is to buy/sell stock
# volume_ratio --> volume_ratio = volume / avg_vol_20 >>> If >1 → today volume is higher than normal, If <1 → lower than normal 
# volume_z     --> how unusual today’s volume is, measured in units of normal variation
#                  (volume - avg_vol_20) / std_vol_20, 
#                  volume - avg_vol_20 >>> How much today’s volume differs from the recent average.
#                  / std_vol_20 >>> normalizes the difference in terms of “typical variation.”
#                  > 0 → volume higher than normal, < 0 → volume lower than normal, ≈ 0 → volume is typical
# liquidity_score ---> measures how unusual today's trading volume is compared to its recent history, It is basically a Z-score of volume, 
# Says > Is today's trading volume normal or unusual?

log("Creating liquidity features...")

cols = ["volume_ratio","volume_z","avg_vol_10","avg_vol_20","liquidity_score"]
for c in cols:
    con.execute(f"ALTER TABLE stocks_features_table DROP COLUMN IF EXISTS {c}")
con.execute("DROP TABLE IF EXISTS temp_liquidity") 


# Creating cols in main table
con.execute("""
    ALTER TABLE stocks_features_table ADD COLUMN volume_ratio DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN volume_z DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN avg_vol_10 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN avg_vol_20 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN liquidity_score DOUBLE;
""")

# Creating tempory temp_liquidity_table for calculate avg_vol_10, avg_vol_20, volume_z, liquidity_score
# Compute rolling averages using a temp table
con.execute("""
    CREATE OR REPLACE TABLE temp_liquidity_table AS
    SELECT 
        symbol,
        date,
        volume,
        std_vol_20,
            AVG(volume) OVER w_10 AS avg_vol_10,
            AVG(volume) OVER w_20 AS avg_vol_20,
            (volume - AVG(volume) OVER w_20) / std_vol_20 AS volume_z,
            volume / AVG(volume) OVER w_20 AS volume_ratio,
            volume / AVG(volume) OVER w_20 * ABS((volume - AVG(volume) OVER w_20) / (std_vol_20 + 1e-9)) AS liquidity_score
    FROM stocks_features_table
    WINDOW
        w_20 AS (PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW),
        w_10 AS (PARTITION BY symbol ORDER BY date ROWS BETWEEN 9 PRECEDING AND CURRENT ROW);
""")

# Overright the newly crated cols in main table with same cols in temp_liquidity_table
con.execute("""
    UPDATE stocks_features_table AS s
    SET
        avg_vol_10 = v.avg_vol_10,
        avg_vol_20 = v.avg_vol_20,
        volume_z = v.volume_z,
        volume_ratio = v.volume_ratio,
        liquidity_score = v.liquidity_score
    FROM temp_liquidity_table AS v
    WHERE s.symbol = v.symbol AND s.date = v.date;
""")

# Overview
df1 = con.execute("SELECT * FROM temp_liquidity_table").fetch_df()
print("4========== temp_liquidity_table \n", df1.head(25),"\n")

df1 = con.execute("SELECT * FROM stocks_features_table").fetch_df()
print("4========== stocks_features_table \n", df1.head(25),"\n")

# tables > temp_liquidity_table
# newly added col > avg_vol_10   avg_vol_20   volume_ratio  volume_z   avg_vol_20   liquidity_score  

Creating liquidity features...
4========== temp_liquidity_table 
         symbol       date  volume   std_vol_20   avg_vol_10   avg_vol_20  \
0   ABAN.N0000 2025-10-23    7508          NaN  7508.000000  7508.000000   
1   ABAN.N0000 2025-10-24    5970          NaN  6739.000000  6739.000000   
2   ABAN.N0000 2025-10-27    2052          NaN  5176.666667  5176.666667   
3   ABAN.N0000 2025-10-28    5545          NaN  5268.750000  5268.750000   
4   ABAN.N0000 2025-10-29    1976          NaN  4610.200000  4610.200000   
5   ABAN.N0000 2025-10-30    1631          NaN  4113.666667  4113.666667   
6   ABAN.N0000 2025-10-31    2734          NaN  3916.571429  3916.571429   
7   ABAN.N0000 2025-11-03    1102          NaN  3564.750000  3564.750000   
8   ABAN.N0000 2025-11-04     962          NaN  3275.555556  3275.555556   
9   ABAN.N0000 2025-11-06    4130          NaN  3361.000000  3361.000000   
10  ABAN.N0000 2025-11-07    2647          NaN  2874.900000  3296.090909   
11  ABAN.N0000 2025-11

### Creating Trend / Technical Features

In [35]:
# =================== Creating Trend / Technical Features =========================================================

log("Creating trend / technical features...")

# Reset tables, cols
cols = ["ma_ratio_5","ma_ratio_10","ma_ratio_20","price_position","breakout_flag"]
for c in cols:
    con.execute(f"ALTER TABLE stocks_features_table DROP COLUMN IF EXISTS {c}")
con.execute("DROP TABLE IF EXISTS temp_trend_table") 



# Add new columns if they don’t exist
con.execute("""
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS ma_ratio_5 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS ma_ratio_10 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS ma_ratio_20 DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS price_position DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS breakout_flag INTEGER;
""")

# Compute moving averages(ma) and rolling highs/lows in a temp table
con.execute("""
    CREATE OR REPLACE TABLE temp_trend_table AS
    SELECT 
        symbol,
        date,
        close,
        high,
        low,
           AVG(close) OVER w_5 AS MA_5,
           AVG(close) OVER w_10 AS MA_10,
           AVG(close) OVER w_20 AS MA_20,
           MAX(high) OVER w_20_prev AS high_20_prev,
           MIN(low) OVER w_20_prev AS low_20_prev
    FROM stocks_features_table
    WINDOW
        w_5 AS (PARTITION BY symbol ORDER BY date ROWS BETWEEN 4 PRECEDING AND CURRENT ROW),
        w_10 AS (PARTITION BY symbol ORDER BY date ROWS BETWEEN 9 PRECEDING AND CURRENT ROW),
        w_20 AS (PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW),
        -- exclude today
        w_20_prev AS (PARTITION BY symbol ORDER BY date ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING);
""")

# Update trend / technical features in the original table
con.execute("""
    UPDATE stocks_features_table AS s
    SET
        ma_ratio_5 = v.close / v.MA_5,
        ma_ratio_10 = v.close / v.MA_10,
        ma_ratio_20 = v.close / v.MA_20,
        price_position = (v.close - v.low_20_prev) / (v.high_20_prev - v.low_20_prev),
        breakout_flag = CASE WHEN v.close > v.high_20_prev THEN 1 ELSE 0 END
    FROM temp_trend_table AS v
    WHERE s.symbol = v.symbol AND s.date = v.date;
""")

df1 = con.execute("SELECT * FROM temp_trend_table").fetch_df()
print("5========== temp_trend_table \n", df1.tail(25),"\n")

df1 = con.execute("SELECT * FROM stocks_features_table").fetch_df()
print("5========== stocks_features_table \n", df1.sample(25),"\n")

# tables > temp_trend_table 
# newly added col > ma_ratio_5   ma_ratio_10   ma_ratio_20   price_position   breakout_flag

Creating trend / technical features...
5========== temp_trend_table 
            symbol       date  close  high   low   MA_5  MA_10   MA_20  \
17427  TRAN.N0000 2026-02-02   57.2  60.0  57.1  57.16  57.24  58.385   
17428  TRAN.N0000 2026-02-03   56.8  59.8  56.5  57.12  57.11  58.200   
17429  TRAN.N0000 2026-02-05   59.4  59.5  59.0  57.60  57.43  58.170   
17430  TRAN.N0000 2026-02-06   57.0  59.0  57.0  57.58  57.43  58.120   
17431  TRAN.N0000 2026-02-09   56.7  57.9  56.0  57.42  57.40  58.055   
17432  TRAN.N0000 2026-02-10   57.9  57.9  57.0  57.56  57.36  57.855   
17433  TRAN.N0000 2026-02-13   57.0  57.0  57.0  57.60  57.36  57.635   
17434  TRAN.N0000 2026-02-16   56.9  57.0  56.9  57.10  57.35  57.480   
17435  TRAN.N0000 2026-02-17   57.5  59.6  56.1  57.20  57.39  57.375   
17436  TRAN.N0000 2026-02-19   56.5  59.6  59.6  57.16  57.29  57.260   
17437  TRAN.N0000 2026-02-20   56.5  56.6  56.5  56.88  57.22  57.230   
17438  TRAN.N0000 2026-02-23   56.5  57.9  56.6  56.78

### Cereating Relative Strength Features

In [36]:
# =================== Cereating Relative Strength Features =========================================================

# # measure how a stock performs relative to its sector
# # ret_5 → stock’s 5-day return
# # avg_sector_ret_5 → 5-day average return of all stocks in the same sector

# log("Creating relative strength features...")

# # Add columns if they don't exist
# con.execute("""
# ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS rel_strength_5 DOUBLE;
# ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS rel_strength_20 DOUBLE;
# """)

# #  Compute sector average returns in a temp table
# con.execute("""
# CREATE OR REPLACE TABLE temp_rel_strength AS
# SELECT *,
#        AVG(ret_5) OVER w_sec_5 AS avg_sector_ret_5,
#        AVG(ret_20) OVER w_sec_20 AS avg_sector_ret_20
# FROM stocks_features_table
# WINDOW
#     w_sec_5  AS (PARTITION BY sector ORDER BY date ROWS BETWEEN 4 PRECEDING AND CURRENT ROW),
#     w_sec_20 AS (PARTITION BY sector ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW);
# """)

# # Update relative strength features in the original table
# con.execute("""
# UPDATE stocks_features_table AS s
# SET
#     rel_strength_5 = v.ret_5 / (v.avg_sector_ret_5 + 1e-9),
#     rel_strength_20 = v.ret_20 / (v.avg_sector_ret_20 + 1e-9)
# FROM temp_rel_strength AS v
# WHERE s.symbol = v.symbol AND s.date = v.date;
# """)

### Creating Other Useful Features

In [37]:
# =================== Creating Other Useful Features =========================================================

"""
TODO
beta
Description: Measures how volatile the stock is relative to the market (or index).
Requires:
Stock returns (ret_1, ret_5, etc.)
Market/index returns (ret_index_1, etc.) → you need an index column or market returns.
Formula:
beta = cov(stock_ret, index_ret) / var(index_ret)
Implementation: Requires sector/index mapping if you want sector beta; otherwise you can use the whole market as reference
"""

log("Creating Other Useful Features...")

# Reset tables, cols
cols = ["momentum_score","volatility_score","turnover_ratio","trend_angle"]
for c in cols:
    con.execute(f"ALTER TABLE stocks_features_table DROP COLUMN IF EXISTS {c}")



# Add new columns
con.execute("""
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS momentum_score DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS volatility_score DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS turnover_ratio DOUBLE;
    ALTER TABLE stocks_features_table ADD COLUMN IF NOT EXISTS trend_angle DOUBLE;
""")

# Update momentum_score, volatility_score, turnover_ratio cols in main table
# Example weights for momentum: 0.5*ret_1 + 0.3*ret_5 + 0.2*ret_10
# Example weights for volatility: 0.5*std_close_5 + 0.3*std_close_10 + 0.2*std_close_20
# Momentum Score   : Weighted sum of returns: 1, 5, 10-day returns
# Volatility Score : Weighted sum of recent standard deviations: std_close_5, std_close_10, std_close_20
# Turnover Ratio   : trades / volume, with 1e-9 to avoid division by zero
# Trend Angle      : Trend Angle measures the direction and strength of a price trend using the slope of a linear regression line 
#                    fitted to the last 20 closing prices. Linear regression slope over a rolling 20-day window per stock (symbol)
#                     \Computed in Pandas because DuckDB doesn’t have rolling regression
con.execute("""
    UPDATE stocks_features_table
    SET
        momentum_score = 0.5 * ret_1 + 0.3 * ret_5 + 0.2 * ret_10,
        volatility_score = 0.5 * std_close_5 + 0.3 * std_close_10 + 0.2 * std_close_20,
        turnover_ratio = trades / (volume + 1e-9)
""")

# Compute trend_angle using Pandas
# DuckDB doesn't have rolling linear regression, so we fetch data into pandas

df_trend_angle = con.execute("SELECT symbol, date, close FROM stocks_features_table ORDER BY symbol, date").df()

trend_window = 20  # number of days for slope

# np.polyfit(x, y, 1) → fits a straight line through the points (x, y); 1 means linear
# [0] → gives the slope (m), which is the Trend Angle
# We apply this function per stock (symbol) because each stock has its own price series
# Positive > Stock is trending upwards (bullish trend)
# Negative > Stock is trending downwards (bearish trend)
# Zero-ish > Stock is sideways, little movement 
def compute_trend_angle(group):
    closes = group['close'].values
    trend_angles = []
    
    for i in range(len(closes)):
        if i < trend_window - 1 or len(closes) < trend_window:
            # Skiooing first 19 records & check not enough data to compute slope → assign NaN
            trend_angles.append(np.nan) 
        else:
            y = closes[i-trend_window+1:i+1]     # last 20 closes
            x = np.arange(trend_window)          # 0,1,2,...19
            slope = np.polyfit(x, y, 1)[0]       # linear regression: slope of best-fit line for values (x, y) 
            trend_angles.append(slope)
    group['trend_angle'] = trend_angles
    return group

df_trend_angle = (
    df_trend_angle
    .groupby('symbol')
    .apply(compute_trend_angle,include_groups=False)
    .reset_index()
)


# Write back trend_angle to DuckDB main table
con.execute("""
    UPDATE stocks_features_table AS s
    SET trend_angle = v.trend_angle
    FROM df_trend_angle AS v
    WHERE s.symbol = v.symbol AND s.date = v.date
""")


df1 = con.execute("SELECT * FROM stocks_features_table").fetch_df()
print("6========== stocks_features_table \n", df1.tail(25),"\n")

# tables > _
# newly added col    > momentum_score    volatility_score   turnover_ratio   trend_angle

Creating Other Useful Features...
6========== stocks_features_table 
               company     symbol  volume  trades  prev_close  open  high  \
17427  VIDULLANKA PLC  VLL.X0000     542       3        17.8  18.3  18.3   
17428  VIDULLANKA PLC  VLL.X0000    4738       9        18.3  17.9  19.0   
17429  VIDULLANKA PLC  VLL.X0000   15035       9        18.9  19.0  19.0   
17430  VIDULLANKA PLC  VLL.X0000    2006       9        18.6  18.9  18.9   
17431  VIDULLANKA PLC  VLL.X0000    4258       8        18.9  18.5  18.5   
17432  VIDULLANKA PLC  VLL.X0000    6152      17        18.5  18.5  18.5   
17433  VIDULLANKA PLC  VLL.X0000    1374       9        18.5  18.5  18.5   
17434  VIDULLANKA PLC  VLL.X0000   10654       5        18.2  18.0  18.6   
17435  VIDULLANKA PLC  VLL.X0000   56311      25        18.4  18.6  18.7   
17436  VIDULLANKA PLC  VLL.X0000   14425      28        18.3  18.5  18.5   
17437  VIDULLANKA PLC  VLL.X0000     154       4        18.0  18.0  18.0   
17438  VIDULLANKA 

### Creating Breakout Context

In [38]:
# =================== Creating Breakout Context =========================================================

log("Creating Breakout Context features...")

# Reset tables, cols
cols = ["range_position"]
for c in cols:
    con.execute(f"ALTER TABLE stocks_features_table DROP COLUMN IF EXISTS {c}")



# Crating range_position col in main table   
con.execute("ALTER TABLE stocks_features_table ADD COLUMN range_position DOUBLE;")

# range_position >>> where the today’s price lies in 20-day range
#                    0 = at 20-day low, 0.5 = middle, 1 = at 20-day high
# detect => breakouts, overbought zones, support/resistance
con.execute("""
    UPDATE stocks_features_table
    SET range_position = (close - low_20) / (high_20 - low_20);
""")

df1 = con.execute("SELECT * FROM stocks_features_table").fetch_df()
print("7========== stocks_features_table after adding v col \n", df1.tail(),"\n")


# =================== Remove Invalid Rows =======================================================================

"""We create a new table so the original data stays safe and unchanged for debugging or reuse.
It also makes the ML pipeline cleaner and reproducible by separating raw data from cleaned data"""
# AT THIS POINT WE CREATE NEW TABLE CALLED "stocks_features_clean_table"
# Creates a cleaned version of the table
# Removes rows where:
#            close_10d IS NULL
#            avg_vol_20 IS NULL
#            std_vol_20 IS NULL
#            std_close_10 IS NULL
#            volume > 0
#            close > 0
# Why cleaning need ? Because early rows don't have enough past data to calculate rolling features

log("Cleaning feature table...")

# Creating stocks_features_clean_table final table
# TODO add more
con.execute("""
CREATE OR REPLACE TABLE stocks_features_clean_table AS
SELECT *
FROM stocks_features_table
WHERE
    close_10d IS NOT NULL
    AND avg_vol_20 IS NOT NULL
    AND std_vol_20 IS NOT NULL
    AND std_close_10 IS NOT NULL
    AND volume > 0
    AND close > 0;
""")


df1 = con.execute("SELECT * FROM stocks_features_table").fetch_df()
print("8========== stocks_features_table after cleaning \n", df1.tail(25),"\n")

# table > stocks_features_clean_table
# newly added col   > range_position

Creating Breakout Context features...
7========== stocks_features_table after adding v col 
               company     symbol  volume  trades  prev_close  open  high  \
17447  VIDULLANKA PLC  VLL.X0000   56783      74        17.3  17.3  17.3   
17448  VIDULLANKA PLC  VLL.X0000   12061      23        16.5  16.9  18.0   
17449  VIDULLANKA PLC  VLL.X0000    9774      19        17.4  17.6  17.6   
17450  VIDULLANKA PLC  VLL.X0000   30359      64        16.7  16.4  16.5   
17451  VIDULLANKA PLC  VLL.X0000    4076       8        15.3  15.0  16.0   

        low  close  change  change_percentage       date  dup_index  close_1d  \
17447  16.0   16.5    -0.8              -4.62 2026-03-09          1      17.3   
17448  16.9   17.4     0.9               5.45 2026-03-10          1      16.5   
17449  16.5   16.5    -0.9              -5.17 2026-03-11          1      17.4   
17450  15.0   15.9    -0.8              -4.79 2026-03-13          1      16.5   
17451  15.0   15.3     0.0               0.00

### Liquidity Ranking

In [39]:
# =================== Liquidity Ranking ==============================================================================

# PARTITION / group stocks by day then order by avg_vol_20 in descending order, Look on a particular date, decide who is #1, #2, etc., based on volume
# Highest volume → rank 1, Equal volumes → same rank, Next rank skips number (because of ties,  > 1, 2, 2, 4),  assign values for each relvent row in the 
# liquidity_rank column in table(stocks_features_clean) by matching symbol with its calculated rank, this doesnt change table row order 
# only the liquidity_rank column, this column is not perfect as 1, 2, 3, may be 3, 1, 2

log("Liquidity Ranking...")

cols = ["liquidity_rank"]
for c in cols:
    con.execute(f"ALTER TABLE stocks_features_clean_table DROP COLUMN IF EXISTS {c}")


con.execute("ALTER TABLE stocks_features_clean_table ADD COLUMN liquidity_rank DOUBLE;")

con.execute("""
    UPDATE stocks_features_clean_table t
    SET liquidity_rank = r.rank
    FROM (
        SELECT
            symbol,
            date,
            RANK() OVER (PARTITION BY date ORDER BY avg_vol_20 DESC) AS rank
        FROM stocks_features_clean_table
    ) r
    WHERE t.symbol = r.symbol
    AND t.date = r.date;
""")


df1 = con.execute("SELECT * FROM stocks_features_clean_table").fetch_df()
print("9========== stocks_features_clean_table \n", df1.tail(25),"\n")

# table > _
# COL   > liquidity_rank

Liquidity Ranking...
9========== stocks_features_clean_table 
               company     symbol  volume  trades  prev_close  open  high  \
11761  VIDULLANKA PLC  VLL.X0000     542       3        17.8  18.3  18.3   
11762  VIDULLANKA PLC  VLL.X0000    4738       9        18.3  17.9  19.0   
11763  VIDULLANKA PLC  VLL.X0000   15035       9        18.9  19.0  19.0   
11764  VIDULLANKA PLC  VLL.X0000    2006       9        18.6  18.9  18.9   
11765  VIDULLANKA PLC  VLL.X0000    4258       8        18.9  18.5  18.5   
11766  VIDULLANKA PLC  VLL.X0000    6152      17        18.5  18.5  18.5   
11767  VIDULLANKA PLC  VLL.X0000    1374       9        18.5  18.5  18.5   
11768  VIDULLANKA PLC  VLL.X0000   10654       5        18.2  18.0  18.6   
11769  VIDULLANKA PLC  VLL.X0000   56311      25        18.4  18.6  18.7   
11770  VIDULLANKA PLC  VLL.X0000   14425      28        18.3  18.5  18.5   
11771  VIDULLANKA PLC  VLL.X0000     154       4        18.0  18.0  18.0   
11772  VIDULLANKA PLC  VL

### Export Feature Dataset

In [40]:
# =================== Export Feature Dataset ============================================================================
# This is the final csv data file used for ML training
log("Exporting feature dataset...")

CSV_FOLDER = "output_csv"
os.makedirs(CSV_FOLDER, exist_ok=True)
FEATURED_CSV_OUTPUT = os.path.join(CSV_FOLDER, "features_master.csv")

con.execute(f"""
    COPY stocks_features_clean_table
    TO '{FEATURED_CSV_OUTPUT}'
    (HEADER, DELIMITER ',');
""")
print("features_master.csv created in output_csv dir\n")

features_master_df = pd.read_csv('output_csv/features_master.csv')


df1 = con.execute("SELECT * FROM stocks_features_clean_table").fetch_df()
print("10========== stocks_features_clean_table \n", df1.tail(),"\n")

print("========== features_master_csv \n", features_master_df.head(25))

Exporting feature dataset...
features_master.csv created in output_csv dir

10========== stocks_features_clean_table 
               company     symbol  volume  trades  prev_close  open  high  \
11781  VIDULLANKA PLC  VLL.X0000   56783      74        17.3  17.3  17.3   
11782  VIDULLANKA PLC  VLL.X0000   12061      23        16.5  16.9  18.0   
11783  VIDULLANKA PLC  VLL.X0000    9774      19        17.4  17.6  17.6   
11784  VIDULLANKA PLC  VLL.X0000   30359      64        16.7  16.4  16.5   
11785  VIDULLANKA PLC  VLL.X0000    4076       8        15.3  15.0  16.0   

        low  close  change  change_percentage       date  dup_index  close_1d  \
11781  16.0   16.5    -0.8              -4.62 2026-03-09          1      17.3   
11782  16.9   17.4     0.9               5.45 2026-03-10          1      16.5   
11783  16.5   16.5    -0.9              -5.17 2026-03-11          1      17.4   
11784  15.0   15.9    -0.8              -4.79 2026-03-13          1      16.5   
11785  15.0   15.3 

### Check the some info of table "stocks_features_clean"

In [41]:
# Shape of the table
print("Shape : ", con.execute("SELECT * FROM stocks_features_table").fetch_df().shape)

# Counts how many rows are missing the ret_10 value.
print("missing for ret_5 : ", con.execute("SELECT COUNT(*) FROM stocks_features_clean_table WHERE ret_5 IS NULL").fetchall())

# Counts how many rows are missing the ret_10 value.
print("missing for ret_10 : ", con.execute("SELECT COUNT(*) FROM stocks_features_clean_table WHERE ret_10 IS NULL").fetchall())

# Finds earliest and latest date in the table.
print("data from-to ", con.execute("SELECT MIN(date), MAX(date) FROM stocks_features_clean_table").fetchall())
print()

stocks_features_clean_df = con.execute("""
SELECT *
    FROM stocks_features_clean_table
""").fetchdf()

stocks_features_clean_df.sample(5)

Shape :  (17452, 49)
missing for ret_5 :  [(0,)]
missing for ret_10 :  [(0,)]
data from-to  [(datetime.date(2025, 12, 31), datetime.date(2026, 3, 19))]



,company,symbol,volume,trades,prev_close,open,high,low,close,change,change_percentage,date,dup_index,close_1d,close_3d,close_5d,close_10d,close_20d,close_60d,high_20,low_20,std_vol_20,ret_1,ret_3,ret_5,ret_10,ret_20,ret_60,std_close_5,std_close_10,std_close_20,range_5,atr_14,TR,volume_ratio,volume_z,avg_vol_10,avg_vol_20,liquidity_score,ma_ratio_5,ma_ratio_10,ma_ratio_20,price_position,breakout_flag,momentum_score,volatility_score,turnover_ratio,trend_angle,range_position,liquidity_rank
2267,WINDFORCE PLC,WIND.N0000,42339,108,47.2,48.00,48.4,46.7,47.0,-0.2,-0.42,2026-02-02,1,47.2,46.6,47.9,48.1,47.80,NaN,48.9,45.90,1.973033e+05,-0.004237,0.008584,-0.018789,-0.022869,-0.016736,NaN,0.511859,0.751960,0.885245,2.8,1.700000,1.7,0.219711,-0.762095,141435.6,192702.90,0.167441,0.995341,0.983058,0.990621,0.305556,0,-0.012329,0.658567,0.002551,0.056917,0.366667,122.0
5375,KELANI VALLEY PLANTATIONS PLC,KVAL.N0000,7199,40,80.0,80.00,80.0,78.1,78.1,-1.9,-2.38,2026-03-19,1,83.4,83.6,83.6,87.2,90.00,NaN,90.1,78.10,1.137557e+04,-0.063549,-0.065789,-0.065789,-0.104358,-0.132222,NaN,2.542243,3.036610,3.202450,8.9,3.050000,5.3,0.722871,-0.242616,11600.6,9958.90,0.175380,0.945979,0.923277,0.898838,-0.104895,0,-0.072383,2.822594,0.005556,-0.482406,0.000000,214.0
6420,THREE ACRE FARMS PLC,TAFL.N0000,20029,81,645.5,640.25,658.0,640.0,641.5,-4.0,-0.62,2026-02-24,1,645.5,646.0,650.0,735.5,757.25,NaN,759.5,640.25,1.325586e+04,-0.006197,-0.006966,-0.013077,-0.127804,-0.152856,NaN,3.250000,31.661019,46.233105,23.0,19.803571,18.0,1.260363,0.312130,15854.3,15891.45,0.393397,0.995731,0.965933,0.911869,0.036585,0,-0.032582,20.369927,0.004044,-7.296241,0.010482,210.0
331,HEMAS HOLDINGS PLC,HHL.N0000,1442806,534,35.5,35.50,36.0,35.0,35.4,-0.1,-0.28,2026-01-08,1,34.8,34.7,34.6,35.3,35.00,NaN,36.2,34.60,3.272561e+06,0.017241,0.020173,0.023121,0.002833,0.011429,NaN,0.311448,0.414327,0.515446,1.6,0.921429,1.2,0.505978,-0.430462,3524975.7,2851519.35,0.217804,1.014908,1.009986,1.006826,0.541667,0,0.016124,0.383111,0.000370,-0.003158,0.500000,13.0
4303,MAHAWELI COCONUT PLANTATIONS PLC,MCPL.N0000,5118,21,53.7,53.00,55.0,52.3,52.5,-1.2,-2.23,2026-02-27,1,53.7,53.2,54.4,56.6,57.70,NaN,56.9,52.50,9.035491e+03,-0.022346,-0.013158,-0.034926,-0.072438,-0.090121,NaN,0.476445,1.231485,1.389661,3.7,2.292857,2.7,0.569157,-0.428781,6697.5,8992.25,0.244044,0.986471,0.972042,0.956633,-0.018519,0,-0.036139,0.885600,0.004103,-0.156692,0.000000,228.0


### Close DuckDB Connection

In [42]:
if 'con' in globals():  # Check if connection exists
    try:
        con.close()        # Close it
        print("DuckDB connection closed.")
    except Exception as e:
        print("Error closing DuckDB:", e)
    finally:
        del con             # Delete the variable from memory

# stocks_features_clean_table

DuckDB connection closed.
